# Live Analyser — CNN-LSTM Predictive Maintenance
### Real-Time Inference from Azure SQL Database

#### Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
from tensorflow.keras.models import load_model
from collections import deque
try:
    import winsound
    SOUND_AVAILABLE = True
except ImportError:
    SOUND_AVAILABLE = False  # Non-Windows fallback

warnings.filterwarnings('ignore')
print('✅ Imports complete.')

✅ Imports complete.


#### Cell 2 — Configuration
> Update Azure credentials and thresholds here.

In [2]:
# ── AZURE SQL CONFIGURATION ─────────────────────────────────────────
AZURE_SERVER   = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM'
AZURE_TABLE    = 'MBP_ControllerData'

# ── MODEL ARTIFACT PATHS ────────────────────────────────────────────
CLASSIFIER_PATH  = 'best_classifier.keras'
AUTOENCODER_PATH = 'best_autoencoder.keras'
SCALER_PATH      = 'scaler.pkl'
ENCODER_PATH     = 'encoder.pkl'
THRESHOLD_PATH   = 'ae_thresholds.pkl'

# ── ANOMALY THRESHOLDS ──────────────────────────────────────────────
# These are set automatically from ae_thresholds.pkl
# You can override manually here after reviewing Cell 14 output
# from CNN_LSTM_Training.ipynb
MANUAL_WARNING_THRESHOLD  = None  # e.g. 0.045 — set to None to use pkl value
MANUAL_CRITICAL_THRESHOLD = None  # e.g. 0.078 — set to None to use pkl value

# ── CLASSIFIER CONFIDENCE THRESHOLD ────────────────────────────────
# Minimum confidence for classifier to report a known breakdown
CLASSIFIER_CONFIDENCE = 0.75

# ── SEQUENCE SETTINGS ───────────────────────────────────────────────
TIME_STEPS = 7  # Must match CNN_LSTM_Training.ipynb

print('✅ Configuration loaded.')

✅ Configuration loaded.


#### Cell 3 — Maintenance Solutions Knowledge Base
> Add solutions for new breakdowns here as data is collected.

In [3]:
SOLUTIONS = {
    'Healthy'           : 'Machine operating within normal parameters. No maintenance required.',
    'Waveness'          : 'Balance gathering. Adjust feed dog height. Check material feed tension. Inspect roller pressure.',
    'High Foot Pressure': 'Adjust the pressure regulator. Check presser foot spring. Calibrate foot pressure setting.',
    'Skip Stitches/Slip': 'Put correct side of needle to needle hole. Replace the needle. Check blade sharpness. Check needle bar height.',
    'Blade Blunt'       : 'Sharpen the blade. Replace blade if sharpening not possible. Check blade alignment after service.',
    # Add more solutions here as new breakdowns are trained
    # 'Needle Breakages' : '...',
    # 'Thread Breakages' : '...',
}

print('✅ Solutions knowledge base loaded.')
print(f'   Known breakdowns: {[k for k in SOLUTIONS if k != "Healthy"]}')

✅ Solutions knowledge base loaded.
   Known breakdowns: ['Waveness', 'High Foot Pressure', 'Skip Stitches/Slip', 'Blade Blunt']


#### Cell 4 — Load Models & Artifacts

In [4]:
try:
    classifier  = load_model(CLASSIFIER_PATH)
    autoencoder = load_model(AUTOENCODER_PATH)

    with open(SCALER_PATH,    'rb') as f: scaler  = pickle.load(f)
    with open(ENCODER_PATH,   'rb') as f: encoder = pickle.load(f)
    with open(THRESHOLD_PATH, 'rb') as f: thresholds = pickle.load(f)

    # Apply manual overrides if set
    WARNING_THRESHOLD  = MANUAL_WARNING_THRESHOLD  or thresholds['warning_threshold']
    CRITICAL_THRESHOLD = MANUAL_CRITICAL_THRESHOLD or thresholds['critical_threshold']

    print('✅ All models and artifacts loaded.')
    print(f'   Classifier classes      : {list(encoder.classes_)}')
    print(f'   Warning threshold  (p95): {WARNING_THRESHOLD:.6f}')
    print(f'   Critical threshold (p99): {CRITICAL_THRESHOLD:.6f}')

except Exception as e:
    print(f'❌ Error loading artifacts: {e}')
    print('   Make sure CNN_LSTM_Training.ipynb has been run first.')

✅ All models and artifacts loaded.
   Classifier classes      : ['Blade Blunt', 'Healthy', 'High Foot Pressure', 'Skip Stitches/Slip', 'Waveness']
   Warning threshold  (p95): 0.819111
   Critical threshold (p99): 1.045780


#### Cell 5 — Feature Extraction (must match training)

In [5]:
def extract_features(df):
    """
    Identical to training feature extraction.
    60 vibration bands + 6 electrical = 66 features.
    """
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts) - 1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except:
            pass
        vib_records.append(vib_dict)

    expected_vib_cols = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df = pd.DataFrame(vib_records).reindex(columns=expected_vib_cols, fill_value=0)

    elec_df = df[[
        'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
        'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax'
    ]].ffill().fillna(0).reset_index(drop=True)

    return pd.concat([vib_df, elec_df], axis=1)

def alert_sound(state):
    """Sound alert — Windows only. Silent fallback on other OS."""
    if not SOUND_AVAILABLE: return
    if state == 'CRITICAL':
        for _ in range(3): winsound.Beep(1000, 500)
    elif state == 'WARNING':
        for _ in range(2): winsound.Beep(800, 400)

print('✅ Feature extractor ready.')

✅ Feature extractor ready.


#### Cell 6 — Live Monitoring Loop
> Inference logic:
> 1. Fetch latest TIME_STEPS records from Azure for selected machine
> 2. Extract features → scale → build sequence
> 3. Run CNN-LSTM Classifier → check confidence
> 4. Run CNN-LSTM Autoencoder → check reconstruction error
> 5. Determine one of 5 states and output result

In [9]:
SELECTED_MACHINE = input('Enter Machine Serial Number (e.g. 0521760): ').strip()
print(f'\n✅ Target machine: {SELECTED_MACHINE}')

driver   = '{ODBC Driver 17 for SQL Server}'
conn_str = (
    f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;'
    f'DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'
)

query = (
    f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE} "
    f"WHERE machineSerialNumber = '{SELECTED_MACHINE}' "
    f"ORDER BY dateTime DESC"
)

last_processed_time = None

try:
    print(f'Connecting to Azure SQL...')
    with pyodbc.connect(conn_str) as conn:
        print(f'✅ Connected. Monitoring machine: {SELECTED_MACHINE}')
        print('=' * 60)

        while True:
            try:
                df_live = pd.read_sql(query, conn)

                # Need exactly TIME_STEPS valid records
                df_live = df_live[
                    df_live['machineVibration'].astype(str).str.startswith('10')
                ]

                if len(df_live) < TIME_STEPS:
                    # Pad by repeating earliest record — handles cold start
                    while len(df_live) < TIME_STEPS:
                        df_live = pd.concat(
                            [df_live.iloc[[0]], df_live], ignore_index=True
                        )
                    df_live = df_live.iloc[:TIME_STEPS]

                db_raw_ts = df_live['dateTime'].iloc[0]

                if db_raw_ts == last_processed_time:
                    time.sleep(1)
                    continue

                last_processed_time = db_raw_ts

                # Time conversion UTC → Sri Lanka (UTC+5:30)
                sl_time      = pd.to_datetime(db_raw_ts) + pd.Timedelta(hours=5, minutes=30)
                current_time = pd.Timestamp.now()

                # Reverse to chronological order (oldest → newest)
                df_window = df_live.iloc[::-1].reset_index(drop=True)

                # Extract features and scale
                features = extract_features(df_window)
                X_scaled = scaler.transform(features.values)
                X_input  = X_scaled.reshape(1, TIME_STEPS, -1)  # (1, 7, 66)

                # ── MODEL 1: CNN-LSTM CLASSIFIER ────────────────────
                cls_probs      = classifier.predict(X_input, verbose=0)[0]
                cls_idx        = np.argmax(cls_probs)
                cls_confidence = cls_probs[cls_idx]
                cls_prediction = encoder.inverse_transform([cls_idx])[0]

                # ── MODEL 2: CNN-LSTM AUTOENCODER ───────────────────
                ae_reconstruction = autoencoder.predict(X_input, verbose=0)
                ae_error          = float(np.mean(np.abs(X_input - ae_reconstruction)))

                # ── STATE DETERMINATION ─────────────────────────────
                # Priority: Classifier (known) > Autoencoder (unknown)
                if cls_prediction != 'Healthy' and cls_confidence >= CLASSIFIER_CONFIDENCE:
                    # Known breakdown detected with high confidence
                    if ae_error < WARNING_THRESHOLD:
                        # Approaching — early warning
                        state     = 'APPROACHING KNOWN BREAKDOWN'
                        state_num = 2
                        icon      = '⚠️ '
                        alert_sound('WARNING')
                    else:
                        # Running with breakdown
                        state     = 'RUNNING WITH KNOWN BREAKDOWN'
                        state_num = 3
                        icon      = '🚨'
                        alert_sound('CRITICAL')

                elif ae_error >= CRITICAL_THRESHOLD:
                    if cls_prediction == 'Healthy':
                        # Autoencoder says critical but classifier says healthy
                        # → Unknown problem running
                        state     = 'RUNNING WITH UNKNOWN PROBLEM'
                        state_num = 5
                        icon      = '🔴'
                        alert_sound('CRITICAL')
                    else:
                        # Both models agree something is very wrong
                        state     = 'RUNNING WITH UNKNOWN PROBLEM'
                        state_num = 5
                        icon      = '🔴'
                        alert_sound('CRITICAL')

                elif ae_error >= WARNING_THRESHOLD:
                    # Approaching unknown problem
                    state     = 'APPROACHING UNKNOWN PROBLEM'
                    state_num = 4
                    icon      = '🟡'
                    alert_sound('WARNING')

                else:
                    # Healthy
                    state     = 'HEALTHY'
                    state_num = 1
                    icon      = '✅'

                # ── OUTPUT ──────────────────────────────────────────
                print('\n' + '=' * 60)
                print(f'AZURE RECORD (UTC)   : {str(db_raw_ts).split(".")[0]}')
                print(f'RECORD TIME (SL)     : {sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'CURRENT LOCAL TIME   : {current_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print('-' * 60)
                print(f'MACHINE STATE        : {icon} {state}')
                print(f'CLASSIFIER           : {cls_prediction} ({cls_confidence*100:.1f}% confidence)')
                print(f'AUTOENCODER ERROR    : {ae_error:.6f}  '
                      f'[warn={WARNING_THRESHOLD:.4f} | crit={CRITICAL_THRESHOLD:.4f}]')
                print('-' * 60)

                if state_num in [2, 3]:
                    print(f'BREAKDOWN            : {cls_prediction}')
                    solution = SOLUTIONS.get(cls_prediction, 'Contact maintenance supervisor.')
                    print(f'SOLUTION             : {solution}')

                elif state_num in [4, 5]:
                    print('BREAKDOWN            : Unknown — not in trained breakdown types')
                    print('ACTION               : Stop machine and contact maintenance supervisor immediately.')

                else:
                    print('No maintenance required.')

                print('=' * 60)

            except Exception as loop_error:
                print(f'⚠️  Loop error: {loop_error}')

            time.sleep(1)

except KeyboardInterrupt:
    print('\n🛑 Monitoring stopped by user.')
except Exception as e:
    print(f'\n❌ Connection error: {e}')


AZURE RECORD (UTC)   : 2026-03-12 11:25:20
RECORD TIME (SL)     : 2026-03-12 16:55:20
CURRENT LOCAL TIME   : 2026-03-12 16:55:22
------------------------------------------------------------
MACHINE STATE        : ✅ HEALTHY
CLASSIFIER           : Healthy (94.7% confidence)
AUTOENCODER ERROR    : 0.564122  [warn=0.8191 | crit=1.0458]
------------------------------------------------------------
No maintenance required.

AZURE RECORD (UTC)   : 2026-03-12 11:25:27
RECORD TIME (SL)     : 2026-03-12 16:55:27
CURRENT LOCAL TIME   : 2026-03-12 16:55:29
------------------------------------------------------------
MACHINE STATE        : ✅ HEALTHY
CLASSIFIER           : Healthy (94.7% confidence)
AUTOENCODER ERROR    : 0.552196  [warn=0.8191 | crit=1.0458]
------------------------------------------------------------
No maintenance required.
⚠️  Loop error: Execution failed on sql: SELECT TOP 7 * FROM MBP_ControllerData WHERE machineSerialNumber = 'YN68884' ORDER BY dateTime DESC
('08S01', '[08S01